# FireSight pilot - option B: fine-tune P50 at 25% data

Continues from the P50 checkpoint (15 epochs, 29,493 images) for **25 more epochs on 25%**
(14,747 images), 640 px, batch 64, AMP. Selection stays on `val.jsonl`; bench_R/TEST are not
in the dataset. Then scores val_cal/val_report at calibration targets 1.5% and 2.0%.


In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available()
print("torch", torch.__version__, "gpus", torch.cuda.device_count())


In [ ]:
import importlib.metadata as md, os
os.environ["YOLO_AUTOINSTALL"] = "False"
os.environ["YOLO_VERBOSE"] = "False"
os.environ["YOLO_OFFLINE"] = "True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
def ver(n):
    try: return md.version(n)
    except Exception: return None
if ver("ultralytics") != "8.4.147":
    !pip install -q --no-deps --progress-bar off ultralytics==8.4.147 ultralytics-thop
print({"ultralytics": ver("ultralytics"), "torch": ver("torch")}, flush=True)


In [ ]:
import os
from pathlib import Path

DATA = str(sorted(Path("/kaggle/input").rglob("manifest_summary.json"))[0].parent)
code_hits = sorted(Path("/kaggle/input").rglob("pilot_train.py"))
pref = [h for h in code_hits if "firesight-pilot-code" in str(h)] or code_hits
CODE = str(pref[0].parent)
assert "def resolve_list" in (Path(CODE) / "pilot_train.py").read_text(), "stale code"
P50 = sorted(Path("/kaggle/input").rglob("P50/weights/best.pt"))[0]
print("DATA", DATA); print("CODE", CODE); print("P50", P50, flush=True)
assert P50.stat().st_size > 10_000_000


In [ ]:
import os, subprocess
WORK = "/kaggle/working"
OUT = f"{WORK}/runs/FT25"
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
cmd = ["timeout", "10800", "python", "-u", f"{CODE}/pilot_train.py", "--dataset", DATA,
       "--fraction", "25", "--model", str(P50), "--epochs", "25", "--imgsz", "640",
       "--batch", "64", "--device", "0", "--workers", "2", "--amp", "--out", OUT]
print("RUN:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, stderr=subprocess.STDOUT, env=env)


In [ ]:
import gc, json, subprocess
import torch
from ultralytics import YOLO
from pathlib import Path

def read_list(name):
    out = []
    for line in open(f"{DATA}/lists/{name}"):
        line = line.strip()
        if line:
            p = Path(line)
            out.append(p if p.is_absolute() else Path(DATA) / p)
    return out

paths = read_list("val_cal.txt") + read_list("val_report.txt")
list_file = f"{WORK}/val_paths.txt"
open(list_file, "w").write("\n".join(str(p) for p in paths) + "\n")
print("val images", len(paths), flush=True)

w = f"{OUT}/weights/best.pt"
model = YOLO(w)
rows = []
for res in model.predict(list_file, imgsz=640, conf=0.01, batch=8, half=True,
                         verbose=False, stream=True):
    dets = [{"label": "fire" if int(b.cls.item()) == 0 else "smoke",
             "confidence": round(float(b.conf.item()), 6),
             "box": [float(v) for v in b.xyxy[0].tolist()]} for b in res.boxes]
    rows.append({"id": Path(res.path).stem, "detections": dets})
preds = f"{WORK}/preds_FT25.jsonl"
open(preds, "w").write("\n".join(json.dumps(r) for r in rows) + "\n")
del model; gc.collect(); torch.cuda.empty_cache()

for target in ("0.015", "0.02"):
    subprocess.run(["python", f"{CODE}/pilot_eval.py", "--dataset", DATA, "--preds", preds,
                    "--model", "FT25", "--target-fpr", target,
                    "--out", f"{WORK}/metrics_FT25_{target}.json"], check=True)


In [ ]:
import json
for t in ("0.015", "0.02"):
    m = json.load(open(f"/kaggle/working/metrics_FT25_{t}.json"))
    c = m["splits"]["val"]["combined"]; f = m["splits"]["val"]["fpr_per_negative_image"]
    print(t, "thr", m["thresholds"], "recall", c["recall"], "ci", c["recall_95ci"],
          "fpr", f["fpr"], "acc", c["accuracy"])
print(open("/kaggle/working/runs/FT25/run_summary.json").read())


Download `runs/FT25/weights/best.pt`, `run_summary.json`, `metrics_FT25_*.json`,
then run the bench_R DEV confirmation locally.
